In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from textblob import TextBlob
from textblob.sentiments import NaiveBayesAnalyzer

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from flair.models import TextClassifier
from flair.data import Sentence

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})
df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [4]:
df = df.sample(2000, random_state=42)
df.shape

(2000, 3)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["label"],
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)

(1600,)
(400,)


In [6]:
def textblob_pattern_predict(text):
    analysis = TextBlob(text)
    return 1 if analysis.sentiment.polarity > 0 else 0

In [7]:
y_pred_tb = X_test.apply(textblob_pattern_predict)
y_pred_tb[:10]

21028    1
46358    1
41470    1
29804    1
34417    1
49476    1
23111    0
17962    1
20304    1
19824    1
Name: review, dtype: int64

In [8]:
def evaluate_model(y_true, y_pred, model_name):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average='macro'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='macro'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='macro'
    )

    print(f"\n===== {model_name} =====")

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    print("\nClassification Report:\n")

    print(classification_report(y_true, y_pred))

In [9]:
evaluate_model(
    y_test,
    y_pred_tb,
    "TextBlob Pattern"
)


===== TextBlob Pattern =====
Accuracy: 0.7025
Precision: 0.780990919857876
Recall: 0.7139582654876124
F1-score: 0.6870047804101815

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.46      0.62       209
           1       0.62      0.97      0.76       191

    accuracy                           0.70       400
   macro avg       0.78      0.71      0.69       400
weighted avg       0.79      0.70      0.68       400



In [10]:
def textblob_nb_predict(text):

    analysis = TextBlob(
        text,
        analyzer=NaiveBayesAnalyzer()
    )

    return 1 if analysis.sentiment.classification == 'pos' else 0

In [11]:
import nltk
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\stephane.holtzmann\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [17]:
y_pred_nb = X_test.apply(textblob_nb_predict)

In [13]:
evaluate_model(
    y_test,
    y_pred_nb,
    "TextBlob NaiveBayes"
)


===== TextBlob NaiveBayes =====
Accuracy: 0.7175
Precision: 0.7642005610098177
Recall: 0.7265086800771563
F1-score: 0.7093453196236922

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.53      0.66       209
           1       0.64      0.93      0.76       191

    accuracy                           0.72       400
   macro avg       0.76      0.73      0.71       400
weighted avg       0.77      0.72      0.71       400



In [14]:
vader = SentimentIntensityAnalyzer()

def vader_predict(text):

    score = vader.polarity_scores(text)

    return 1 if score['compound'] >= 0 else 0

In [15]:
y_pred_vader = X_test.apply(vader_predict)

In [16]:
evaluate_model(
    y_test,
    y_pred_vader,
    "VADER"
)


===== VADER =====
Accuracy: 0.7275
Precision: 0.7536892361111112
Recall: 0.7342744056714847
F1-score: 0.7236851277956284

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.58      0.69       209
           1       0.66      0.88      0.76       191

    accuracy                           0.73       400
   macro avg       0.75      0.73      0.72       400
weighted avg       0.76      0.73      0.72       400



In [18]:
classifier = TextClassifier.load('sentiment')

2026-05-20 16:16:36,360 https://nlp.informatik.hu-berlin.de/resources/models/sentiment-curated-distilbert/sentiment-en-mix-distillbert_4.pt not found in cache, downloading to C:\Users\STEPHA~1.HOL\AppData\Local\Temp\tmp4cmoh0ta


100%|███████████████████████████████████████████████████████████████████████████████| 253M/253M [00:07<00:00, 37.0MB/s]

2026-05-20 16:16:43,897 copying C:\Users\STEPHA~1.HOL\AppData\Local\Temp\tmp4cmoh0ta to cache at C:\Users\stephane.holtzmann\.flair\models\sentiment-en-mix-distillbert_4.pt


2026-05-20 16:16:44,005 removing temp file C:\Users\STEPHA~1.HOL\AppData\Local\Temp\tmp4cmoh0ta


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
def flair_predict(text):

    sentence = Sentence(text)

    classifier.predict(sentence)

    label = sentence.labels[0]

    return 1 if label.value == 'POSITIVE' else 0

In [20]:
y_pred_flair = X_test.apply(flair_predict)

In [21]:
evaluate_model(
    y_test,
    y_pred_flair,
    "Flair"
)


===== Flair =====
Accuracy: 0.9025
Precision: 0.9117920628358827
Recall: 0.8992584984593802
F1-score: 0.9012501978161102

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.97      0.91       209
           1       0.96      0.83      0.89       191

    accuracy                           0.90       400
   macro avg       0.91      0.90      0.90       400
weighted avg       0.91      0.90      0.90       400

